# 🐍 `asyncio` — Complete Learning & Practice Workbook

## What is `asyncio`?
`asyncio` is Python's built-in library for writing **concurrent code** using the **async/await** syntax.

It is based on the **event loop** model — instead of running things in parallel threads, a single thread cooperatively switches between tasks whenever one is *waiting* (e.g., for network I/O, file I/O, a timer).

---

## Why does it exist?

Traditional blocking code wastes CPU time waiting:
```
Thread 1: ──[fetch data]──────────────────[process]──
                          ↑ CPU sits idle here
```

Async code reuses that idle time:
```
Event Loop: ──[start fetch A]──[start fetch B]──[A done → process A]──[B done → process B]──
```

---

## Core concepts covered in this workbook

| # | Concept | Purpose |
|---|---------|---------|
| 1 | `async def` / `await` | Define and call coroutines |
| 2 | Event Loop | The engine driving async execution |
| 3 | `asyncio.gather()` | Run multiple coroutines concurrently |
| 4 | `asyncio.create_task()` | Schedule background tasks |
| 5 | Timeouts & Cancellation | `asyncio.wait_for`, `task.cancel()` |
| 6 | `asyncio.Queue` | Producer/Consumer pattern |
| 7 | Semaphores | Limit concurrency |
| 8 | Real-world: Async HTTP | `aiohttp` for concurrent API calls |


---
## Module 1 — Coroutines: `async def` and `await`

### Theory
A **coroutine** is a special function defined with `async def`. It does NOT run when called — it returns a coroutine object. You must `await` it to actually execute it.

- `await` = "pause here and let the event loop do other things while I wait"
- Only valid **inside** an `async def` function
- In Jupyter, the top-level event loop is already running, so you can `await` directly in cells

**Analogy:** Imagine ordering coffee. Normal (sync) code stands at the counter and waits. Async code places the order, sits down, does other things, and comes back when the coffee is ready.


In [3]:
import asyncio
import time

start_time = time.perf_counter()

def ts():
    """Returns elapsed seconds since cell started — so you can see the pause."""
    return f"[t={time.perf_counter() - start_time:.1f}s]"

# Define a coroutine — notice 'async def'
async def make_coffee(name: str, seconds: float) -> str:
    print(f"  {ts()} ☕ Starting {name} order...")
    await asyncio.sleep(seconds)   # non-blocking wait (simulates I/O delay)
    print(f"  {ts()} ✅ {name} is ready! (took {seconds}s)")
    return f"{name} done"

# --- Single coroutine: you can see the pause in the timestamps ---
print(f"{ts()} Running make_coffee sequentially...")
result = await make_coffee("Espresso", 1.0)   # reduced from 10s to 1s
print(f"{ts()} Return value: {result}")

# --- Now run TWO concurrently with gather — both start at the same time ---
print(f"\n{ts()} Running two coffees CONCURRENTLY with gather...")
r1, r2 = await asyncio.gather(
    make_coffee("Latte",     seconds=1.5),
    make_coffee("Espresso",  seconds=0.8),
)
# Notice: Espresso (0.8s) finishes BEFORE Latte (1.5s)
# Total time ≈ 1.5s, NOT 1.5+0.8=2.3s  ← THIS is where async shines
print(f"{ts()} Both done! Results: {r1}, {r2}")

# Calling without await — shows it's just a coroutine object, not executed
coro = make_coffee("Latte", 1.0)
print(f"\nWithout await: {coro}")   # <coroutine object ...>
coro.close()  # clean up the unawaited coroutine to avoid warnings


[t=0.0s] Running make_coffee sequentially...
  [t=0.0s] ☕ Starting Espresso order...
  [t=1.0s] ✅ Espresso is ready! (took 1.0s)
[t=1.0s] Return value: Espresso done

[t=1.0s] Running two coffees CONCURRENTLY with gather...
  [t=1.0s] ☕ Starting Latte order...
  [t=1.0s] ☕ Starting Espresso order...
  [t=1.0s] ✅ Espresso is ready! (took 1.0s)
[t=1.0s] Return value: Espresso done

[t=1.0s] Running two coffees CONCURRENTLY with gather...
  [t=1.0s] ☕ Starting Latte order...
  [t=1.0s] ☕ Starting Espresso order...
  [t=1.8s] ✅ Espresso is ready! (took 0.8s)
  [t=1.8s] ✅ Espresso is ready! (took 0.8s)
  [t=2.5s] ✅ Latte is ready! (took 1.5s)
[t=2.5s] Both done! Results: Latte done, Espresso done

Without await: <coroutine object make_coffee at 0x0000025C88E86B40>
  [t=2.5s] ✅ Latte is ready! (took 1.5s)
[t=2.5s] Both done! Results: Latte done, Espresso done

Without await: <coroutine object make_coffee at 0x0000025C88E86B40>


---
## Module 2 — The Event Loop

### Theory
The **event loop** is the core engine of `asyncio`. It:
1. Maintains a queue of coroutines/tasks to run
2. Runs each one until it hits an `await`
3. Suspends it, runs another ready task
4. Resumes the suspended task when its awaited work is done

**Key rule:** There is only **one event loop per thread**. Jupyter notebooks already run an event loop internally — that's why you can `await` directly in cells without `asyncio.run()`.

In a regular Python script (not Jupyter), you start the loop with:
```python
asyncio.run(my_coroutine())
```

Below we inspect the running loop and see its internals.


In [4]:
import asyncio

# Get the currently running event loop (Jupyter has one running already)
loop = asyncio.get_event_loop()
print("Loop type      :", type(loop).__name__)
print("Loop running?  :", loop.is_running())
print("Loop closed?   :", loop.is_closed())

# Demonstrate sequential vs concurrent timing
import time

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

# Sequential — total time = sum of all delays
start = time.perf_counter()
r1 = await task("A", 0.5)
r2 = await task("B", 0.5)
r3 = await task("C", 0.5)
print(f"\nSequential results: {r1}, {r2}, {r3}")
print(f"Sequential time: {time.perf_counter() - start:.2f}s  (expected ~1.5s)")


Loop type      : _WindowsSelectorEventLoop
Loop running?  : True
Loop closed?   : False

Sequential results: A, B, C
Sequential time: 1.51s  (expected ~1.5s)

Sequential results: A, B, C
Sequential time: 1.51s  (expected ~1.5s)


---
## Module 3 — `asyncio.gather()`: Running Tasks Concurrently

### Theory
`asyncio.gather(*coroutines)` schedules **all coroutines at once** and waits for all of them to complete. It returns a list of results in the same order as the inputs.

- All tasks start **immediately** and run concurrently
- Total time ≈ the **slowest** single task (not the sum)
- If any coroutine raises an exception, it propagates (use `return_exceptions=True` to collect errors instead)

**When to use:** Fetching data from multiple APIs, reading multiple files, running independent computations.


In [ ]:
import asyncio
import time

async def fetch_data(source: str, delay: float) -> dict:
    """Simulates fetching data from a remote source."""
    print(f"  → Fetching from {source}...")
    await asyncio.sleep(delay)
    print(f"  ← {source} responded after {delay}s")
    return {"source": source, "data": f"payload_from_{source}"}

# --- Concurrent with gather ---
print("=== asyncio.gather() — concurrent ===")
start = time.perf_counter()

results = await asyncio.gather(
    fetch_data("GitHub API",    delay=1.0),
    fetch_data("Database",      delay=0.8),
    fetch_data("Config Server", delay=0.5),
)

elapsed = time.perf_counter() - start
print(f"\nAll done in {elapsed:.2f}s  (expected ~1.0s, not ~2.3s)")
print("\nResults:")
for r in results:
    print(" ", r)

# --- gather with return_exceptions=True ---
print("\n=== gather with return_exceptions=True ===")

async def might_fail(name, should_fail=False):
    await asyncio.sleep(0.2)
    if should_fail:
        raise ValueError(f"{name} failed!")
    return f"{name} OK"

results = await asyncio.gather(
    might_fail("Task A"),
    might_fail("Task B", should_fail=True),
    might_fail("Task C"),
    return_exceptions=True   # errors become values, don't crash gather
)

for r in results:
    if isinstance(r, Exception):
        print(f"  ❌ Error: {r}")
    else:
        print(f"  ✅ {r}")


---
## Module 4 — `asyncio.create_task()`: Fire-and-Forget Background Tasks

### Theory
`asyncio.create_task(coroutine)` wraps a coroutine into a **Task** and schedules it to run on the event loop **immediately in the background** — without blocking the current coroutine.

**Difference from `gather`:**
| | `gather` | `create_task` |
|---|---|---|
| Waits for results? | Yes, blocks until all done | No — runs in background |
| Use case | Need all results before continuing | Fire-and-forget, or await later |
| Returns | List of results | `Task` object |

Tasks have a lifecycle: **pending → running → done/cancelled**. You can check status with `.done()`, `.cancelled()`, `.result()`.


In [ ]:
import asyncio

async def background_job(name: str, delay: float):
    await asyncio.sleep(delay)
    print(f"  [background] {name} finished after {delay}s")
    return f"{name}_result"

async def main_work():
    print("Main: starting background tasks...")

    # Schedule tasks — they start immediately but we don't wait yet
    task1 = asyncio.create_task(background_job("Logger", 1.0))
    task2 = asyncio.create_task(background_job("Metrics", 0.6))

    print(f"Main: task1 done? {task1.done()}")   # False — still running

    print("Main: doing other work while tasks run in background...")
    await asyncio.sleep(0.3)   # simulating main work
    print("Main: still working...")
    await asyncio.sleep(0.5)

    # Now wait for tasks to complete and get results
    result1 = await task1
    result2 = await task2

    print(f"\nMain: task1 done? {task1.done()}")  # True now
    print(f"Results: {result1}, {result2}")

await main_work()


---
## Module 5 — Timeouts and Cancellation

### Theory
In real systems, you can't wait forever. `asyncio` provides two tools:

- **`asyncio.wait_for(coro, timeout)`** — raises `asyncio.TimeoutError` if the coroutine doesn't finish in time. It also **cancels** the coroutine automatically.
- **`task.cancel()`** — sends a `CancelledError` into the coroutine at the next `await` point. The coroutine can catch it to do cleanup.

**Why cancellation matters:** If you're fetching 10 URLs and one hangs forever, you need a way to abort it cleanly without killing your whole program.


In [ ]:
import asyncio

async def slow_operation(name: str, duration: float):
    print(f"  {name}: starting (will take {duration}s)...")
    try:
        await asyncio.sleep(duration)
        print(f"  {name}: completed!")
        return f"{name} result"
    except asyncio.CancelledError:
        print(f"  {name}: ⚠️  was cancelled! Running cleanup...")
        # Always re-raise CancelledError after cleanup
        raise

# --- wait_for: automatic timeout ---
print("=== asyncio.wait_for() ===")
try:
    result = await asyncio.wait_for(slow_operation("FastTask", 0.3), timeout=1.0)
    print("Result:", result)
except asyncio.TimeoutError:
    print("Timed out!")

print()
try:
    result = await asyncio.wait_for(slow_operation("SlowTask", 2.0), timeout=0.5)
    print("Result:", result)
except asyncio.TimeoutError:
    print("SlowTask timed out after 0.5s ❌")

# --- Manual task cancellation ---
print("\n=== task.cancel() ===")
task = asyncio.create_task(slow_operation("ManualCancel", 3.0))

await asyncio.sleep(0.5)   # let task start
print("Cancelling task...")
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print(f"Task cancelled. task.cancelled() = {task.cancelled()}")


---
## Module 6 — `asyncio.Queue`: Producer / Consumer Pattern

### Theory
`asyncio.Queue` is a thread-safe, async-friendly FIFO queue used for the **Producer/Consumer** pattern:

- **Producers** put items into the queue (`await queue.put(item)`)
- **Consumers** take items out (`await queue.get()` — blocks if empty)
- `queue.task_done()` signals that a consumed item has been processed
- `queue.join()` blocks until all items have been processed

**Use case:** Processing a stream of work items (e.g., URLs to scrape, jobs from a message broker) with a fixed pool of workers — prevents spawning thousands of tasks at once.

```
Producer ──→ [Queue] ──→ Worker 1
                    ──→ Worker 2
                    ──→ Worker 3
```


In [ ]:
import asyncio
import random

async def producer(queue: asyncio.Queue, n_items: int):
    """Produces n_items work items and puts them in the queue."""
    for i in range(1, n_items + 1):
        job = f"job_{i:02d}"
        await queue.put(job)
        print(f"  📦 Producer: queued {job}  (queue size: {queue.qsize()})")
        await asyncio.sleep(0.1)   # simulate work between producing
    print("  📦 Producer: all jobs queued")

async def worker(worker_id: int, queue: asyncio.Queue):
    """Continuously pulls jobs from the queue and processes them."""
    while True:
        job = await queue.get()   # blocks if queue is empty
        process_time = random.uniform(0.2, 0.5)
        print(f"  🔧 Worker-{worker_id}: processing {job} ({process_time:.2f}s)")
        await asyncio.sleep(process_time)
        print(f"  ✅ Worker-{worker_id}: finished {job}")
        queue.task_done()   # IMPORTANT: signal that this item is processed

async def run_pipeline():
    queue = asyncio.Queue(maxsize=5)   # maxsize limits buffer; producer blocks if full

    # Start 3 workers as background tasks
    workers = [asyncio.create_task(worker(i, queue)) for i in range(1, 4)]

    # Run producer
    await producer(queue, n_items=8)

    # Wait until all items in the queue are processed
    await queue.join()
    print("\n🎉 All jobs processed!")

    # Cancel idle workers (they're stuck waiting on an empty queue)
    for w in workers:
        w.cancel()

await run_pipeline()


---
## Module 7 — `asyncio.Semaphore`: Limiting Concurrency

### Theory
A **Semaphore** is a concurrency limiter. It holds an internal counter:
- `await semaphore.acquire()` → decrements counter; **blocks if counter is 0**
- `semaphore.release()` → increments counter

Used as an `async with` context manager, it ensures **at most N coroutines run a section of code simultaneously**.

**Real-world use case:** You want to send 100 HTTP requests concurrently, but the API rate-limits you to 5 simultaneous connections. A `Semaphore(5)` enforces this automatically.

Without semaphore → 100 tasks flood the API → rate limit errors  
With `Semaphore(5)` → max 5 active at any time → polite, controlled concurrency


In [ ]:
import asyncio
import time

active_count = 0   # track how many tasks are running simultaneously

async def api_call(sem: asyncio.Semaphore, request_id: int):
    global active_count
    async with sem:   # blocks here if semaphore counter == 0
        active_count += 1
        print(f"  Request {request_id:02d} started  | active now: {active_count}")
        await asyncio.sleep(0.4)   # simulate API call
        active_count -= 1
        print(f"  Request {request_id:02d} finished | active now: {active_count}")
        return f"response_{request_id}"

# --- Without semaphore: all 10 launch at once ---
print("=== No Semaphore — all 10 at once ===")
start = time.perf_counter()
results = await asyncio.gather(*[api_call(asyncio.Semaphore(10), i) for i in range(1, 11)])
print(f"Done in {time.perf_counter() - start:.2f}s\n")

# --- With Semaphore(3): max 3 at a time ---
active_count = 0
print("=== Semaphore(3) — max 3 concurrent ===")
sem = asyncio.Semaphore(3)
start = time.perf_counter()
results = await asyncio.gather(*[api_call(sem, i) for i in range(1, 11)])
print(f"Done in {time.perf_counter() - start:.2f}s  (should be ~ceil(10/3)*0.4 ≈ 1.6s)")
print(f"Got {len(results)} results")


---
## Module 8 — `asyncio.Event` and `asyncio.Lock`: Synchronisation Primitives

### Theory

**`asyncio.Event`** — a simple flag that coroutines can wait on:
- `event.set()` → signals all waiters
- `event.clear()` → resets the flag
- `await event.wait()` → blocks until the event is set

**`asyncio.Lock`** — a mutual exclusion lock:
- Only one coroutine can hold the lock at a time
- Prevents **race conditions** when multiple coroutines modify shared state
- Used as `async with lock:`

These are async equivalents of `threading.Event` and `threading.Lock`, but designed to work within the event loop without blocking it.


In [ ]:
import asyncio

# --- asyncio.Event: signal between coroutines ---
print("=== asyncio.Event ===")

async def waiter(event: asyncio.Event, name: str):
    print(f"  {name}: waiting for event...")
    await event.wait()
    print(f"  {name}: event received! Proceeding.")

async def trigger(event: asyncio.Event):
    await asyncio.sleep(0.8)
    print("  Trigger: setting event!")
    event.set()

event = asyncio.Event()
await asyncio.gather(
    waiter(event, "Waiter-1"),
    waiter(event, "Waiter-2"),
    trigger(event),
)

# --- asyncio.Lock: protecting shared state ---
print("\n=== asyncio.Lock — Race condition demo ===")

shared_counter = 0
lock = asyncio.Lock()

async def increment_unsafe(n: int):
    """WITHOUT lock — race condition possible."""
    global shared_counter
    for _ in range(n):
        val = shared_counter
        await asyncio.sleep(0)   # yield control — another task may run here!
        shared_counter = val + 1

async def increment_safe(n: int):
    """WITH lock — atomically read-modify-write."""
    global shared_counter
    for _ in range(n):
        async with lock:
            val = shared_counter
            await asyncio.sleep(0)
            shared_counter = val + 1

# Unsafe: run 3 concurrent tasks, each incrementing 5 times = should be 15
shared_counter = 0
await asyncio.gather(increment_unsafe(5), increment_unsafe(5), increment_unsafe(5))
print(f"Unsafe result:  {shared_counter} (expected 15, but may be wrong due to race!)")

# Safe: same but with lock
shared_counter = 0
await asyncio.gather(increment_safe(5), increment_safe(5), increment_safe(5))
print(f"Safe result:    {shared_counter} (always exactly 15 ✅)")


---
## Module 9 — Real-World Pattern: Async HTTP with `aiohttp`

### Theory
`requests` is a **blocking** library — while it waits for a server response, it blocks the entire event loop. For async code you need an async HTTP client.

**`aiohttp`** is the standard choice:
- `aiohttp.ClientSession` is the async equivalent of `requests.Session`
- All I/O operations are `await`-able
- Combine with `asyncio.gather()` + `Semaphore` for high-throughput concurrent requests

**Pattern:**
```python
async with aiohttp.ClientSession() as session:
    async with session.get(url) as response:
        data = await response.json()
```

> 💡 If `aiohttp` is not installed, the next cell will install it automatically.


In [ ]:
import subprocess, sys
try:
    import aiohttp
    print("aiohttp already installed:", aiohttp.__version__)
except ImportError:
    print("Installing aiohttp...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "aiohttp", "-q"])
    import aiohttp
    print("aiohttp installed:", aiohttp.__version__)


In [ ]:
import asyncio
import time
import aiohttp

# Fetch multiple GitHub API endpoints concurrently using aiohttp + Semaphore
URLS = [
    "https://api.github.com",
    "https://api.github.com/zen",          # returns a random GitHub "zen" quote
    "https://api.github.com/octocat",      # returns ASCII art
    "https://api.github.com/rate_limit",   # shows your current rate limit
]

async def fetch(session: aiohttp.ClientSession, sem: asyncio.Semaphore, url: str) -> dict:
    async with sem:
        print(f"  → GET {url}")
        async with session.get(url) as resp:
            status = resp.status
            content_type = resp.headers.get("Content-Type", "")
            if "json" in content_type:
                body = await resp.json()
                preview = str(body)[:80] + "..."
            else:
                body = await resp.text()
                preview = body[:80]
            print(f"  ← {status} {url.split('/')[-1] or 'root'}")
            return {"url": url, "status": status, "preview": preview}

async def fetch_all(urls: list[str]):
    sem = asyncio.Semaphore(3)
    headers = {"Accept": "application/vnd.github+json"}

    async with aiohttp.ClientSession(headers=headers) as session:
        tasks = [fetch(session, sem, url) for url in urls]
        return await asyncio.gather(*tasks, return_exceptions=True)

start = time.perf_counter()
results = await fetch_all(URLS)
elapsed = time.perf_counter() - start

print(f"\n✅ Fetched {len(results)} URLs in {elapsed:.2f}s\n")
for r in results:
    if isinstance(r, Exception):
        print(f"  ❌ Error: {r}")
    else:
        print(f"  [{r['status']}] {r['url']}")
        print(f"       {r['preview']}\n")


---
## Module 10 — Quick Reference & Common Pitfalls

### ✅ Cheat Sheet

```python
# Define a coroutine
async def my_func(): ...

# Await a coroutine (inside async context)
result = await my_func()

# Run from regular (non-async) script entry point
asyncio.run(my_func())

# Run multiple concurrently, collect results
results = await asyncio.gather(coro1(), coro2(), coro3())

# Schedule background task (don't wait immediately)
task = asyncio.create_task(my_func())
result = await task   # wait for it later

# Timeout
result = await asyncio.wait_for(my_func(), timeout=5.0)

# Limit concurrency
sem = asyncio.Semaphore(5)
async with sem:
    await do_work()

# Queue-based pipeline
q = asyncio.Queue()
await q.put(item)
item = await q.get()
q.task_done()
await q.join()

# Non-blocking sleep (ALWAYS use this, not time.sleep!)
await asyncio.sleep(1.0)
```

---

### ❌ Common Pitfalls

| Mistake | Problem | Fix |
|---------|---------|-----|
| `time.sleep(1)` inside async | Blocks the entire event loop | Use `await asyncio.sleep(1)` |
| Forgetting `await` on a coroutine | Coroutine never runs, no error | Always `await` coroutine calls |
| `asyncio.run()` inside Jupyter | RuntimeError: loop already running | Just `await` directly in Jupyter |
| Not calling `task_done()` after `queue.get()` | `queue.join()` hangs forever | Always call `queue.task_done()` |
| Catching `CancelledError` without re-raising | Task can't be properly cancelled | Always `raise` after cleanup |
| CPU-heavy work in async | Blocks event loop even with `await` | Use `loop.run_in_executor()` for CPU tasks |

---

### 🏁 Practice Exercises

1. **Retry logic** — Write an async function that retries a failing HTTP request up to 3 times with exponential backoff using `asyncio.sleep`.
2. **Rate limiter** — Use a `Semaphore` + `asyncio.Queue` to fetch 50 URLs with max 5 concurrent connections.
3. **Pub/Sub** — Use `asyncio.Event` to build a simple pub/sub: one publisher sets an event, multiple subscribers react.
4. **Timeout pipeline** — Wrap each `gather` task in `wait_for` with a 1s timeout; collect successes and log timeouts.
